<a href="https://colab.research.google.com/github/nonthawat2548/lab-ai-69/blob/main/ThaiFoodAI_THFOOD50_FastTraining_AutoOnly.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🍲 ThaiFood AI — THFOOD-50 + RAG (ฝึกเร็วขึ้น • วิเคราะห์ภาพอัตโนมัติ)
**Google Colab Notebook ไฟล์เดียว** ใช้ภาพจาก Hugging Face ครบ 50 คลาส และไม่มีเมนูเลือกหรือแก้ชื่ออาหารด้วยตนเอง

### ทำไมเวอร์ชันนี้จึงฝึกได้เร็วกว่าเดิม
โน้ตบุ๊กเดิมใช้ Streaming อ่านภาพจาก Hugging Face **ใหม่ทุก Epoch** และให้ EfficientNet ประมวลผลภาพทั้งหมดซ้ำ 6 รอบ แม้ส่วนสกัดคุณลักษณะจะถูก Freeze ไว้

เวอร์ชันนี้จึงเปลี่ยนเป็น **Cached Feature Transfer Learning**:
1. ดาวน์โหลด Dataset มาเก็บในแคช Colab **ครั้งเดียว** (การดาวน์โหลดครั้งแรกอาจใช้เวลาตามความเร็วเน็ต)
2. ให้ EfficientNet-B0 สกัดคุณลักษณะภาพ Train/Validation **เพียงหนึ่งรอบ** ด้วย GPU และ Mixed Precision
3. เก็บเวกเตอร์คุณลักษณะ แล้วฝึกเฉพาะชั้นจำแนกบนเวกเตอร์เหล่านั้น โดย **ไม่เปิดภาพและไม่ประมวลผล CNN ใหม่ในทุก Epoch**
4. บันทึกโมเดลที่ดีที่สุดตาม Validation Macro F1 และใช้ Test Set แยกต่างหาก

> จำนวน Epoch ของชั้นจำแนกอาจมากกว่าเดิมได้ เพราะเป็นการฝึกกับเวกเตอร์ขนาดเล็ก ไม่เทียบเวลา Epoch ต่อ Epoch กับการฝึก CNN เต็มรูปแบบ; เวลาจริงขึ้นกับ Colab runtime และการดาวน์โหลดข้อมูล

**ข้อจำกัด:** วิธีนี้เป็น Fast Baseline ที่ใช้ Transfer Learning แบบ Freeze Backbone ผลลัพธ์อาจไม่แม่นเท่าการ Fine-tune ชั้นภาพ จึงต้องตรวจ Accuracy/Macro F1 จริงก่อนใช้งาน หากจำแนกผิด ไม่ควรอ้างว่าคะแนน Softmax คือความแม่นยำ


## 0. ติดตั้ง และเตรียมเครื่อง
Runtime → Change runtime type → **T4 GPU** (ถ้ามี) แล้วรันเซลล์ติดตั้งหนึ่งครั้ง

In [1]:

%pip -q install "datasets>=3.0,<5" "huggingface_hub>=1.16,<3" "gradio>=6.30,<7" "scikit-learn>=1.5" "pandas==2.2.3" "google-genai>=2.12.1,<3" "starlette>=1.3.1,<2" "pymupdf>=1.24" "matplotlib"


In [2]:
import os, io, re, json, random, math, hashlib
from IPython.display import display
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd
import torch
from torch import nn
from torch.utils.data import IterableDataset, DataLoader
from torchvision import transforms, models
from PIL import Image, ImageOps
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

SEED = 2026
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
ROOT = Path('/content/ThaiFoodAI_THFOOD50') if Path('/content').exists() else Path.cwd()/'ThaiFoodAI_THFOOD50'
ROOT.mkdir(exist_ok=True,parents=True)
HF_DATASET = 'thean/THFOOD-50'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('DEVICE:', DEVICE, '| FILES:', ROOT)

DEVICE: cuda | FILES: /content/ThaiFoodAI_THFOOD50


## 1. ชื่อเมนูครบ 50 คลาส และวัตถุดิบทั่วไป
ชื่อคลาส **ต้องตรงกับ label ใน Hugging Face** ทั้งตัวสะกด รวมถึงชื่อที่ Dataset สะกดแปลก ๆ เช่น `GrilledQquid` และ `KkaoKlukKaphi`  รายการส่วนผสมเป็นข้อมูลประกอบเชิงตำรับทั่วไป มีความแตกต่างตามสูตรและร้าน; **ไม่ใช่ฐานข้อมูลส่วนประกอบที่รับรองโดยกรมอนามัย**

In [3]:
# คีย์ต้องตรงกับ labels ของ THFOOD-50 (เรียงโดย class index)
MENU_METADATA = {
 'BitterMelonSoup':('ต้มมะระยัดไส้','มะระ, หมูสับ, วุ้นเส้น (บางสูตร), กระเทียม, พริกไทย, น้ำซุป'),
 'BooPadPongali':('ปูผัดผงกะหรี่','เนื้อปู, ผงกะหรี่, ไข่, นม, หอมหัวใหญ่, ต้นหอม, น้ำมัน'),
 'CurriedFishCake':('ทอดมันปลา','เนื้อปลา, พริกแกงแดง, ถั่วฝักยาว, ใบมะกรูด, ไข่, น้ำมันทอด'),
 'Dumpling':('ขนมจีบ','แผ่นเกี๊ยว, หมูหรือกุ้งสับ, กระเทียม, พริกไทย, เครื่องปรุง'),
 'EggsStewed':('ไข่พะโล้','ไข่, หมู, เต้าหู้ (บางสูตร), โป๊ยกั๊ก, อบเชย, ซีอิ๊ว'),
 'FriedChicken':('ไก่ทอด','ไก่, กระเทียม, พริกไทย, เกลือ, แป้ง (บางสูตร), น้ำมันทอด'),
 'FriedKale':('ผัดคะน้า','คะน้า, กระเทียม, น้ำมัน, ซอสปรุงรส, เนื้อสัตว์ (บางสูตร)'),
 'FriedMusselPancakes':('หอยทอด','หอยแมลงภู่, แป้ง, ไข่, ถั่วงอก, ต้นหอม, น้ำมัน'),
 'GaengJued':('แกงจืด','น้ำซุป, ผักกาดขาวหรือสาหร่าย, เต้าหู้, หมูสับ, ต้นหอม'),
 'GaengKeawWan':('แกงเขียวหวาน','กะทิ, พริกแกงเขียวหวาน, เนื้อสัตว์, มะเขือ, ใบโหระพา'),
 'GaiYang':('ไก่ย่าง','ไก่, กระเทียม, รากผักชี, พริกไทย, น้ำปลา, เครื่องหมัก'),
 'GoongObWoonSen':('กุ้งอบวุ้นเส้น','กุ้ง, วุ้นเส้น, ขิง, กระเทียม, พริกไทย, ซอสปรุงรส'),
 'GoongPao':('กุ้งเผา','กุ้ง, น้ำจิ้มซีฟู้ด (พริก, กระเทียม, มะนาว, น้ำปลา)'),
 'GrilledQquid':('ปลาหมึกย่าง','ปลาหมึก, น้ำจิ้มซีฟู้ด, กระเทียม, มะนาว'),
 'HoyKraeng':('หอยแครงลวก','หอยแครง, น้ำจิ้มซีฟู้ด, มะนาว, พริก, กระเทียม'),
 'HoyLaiPrikPao':('หอยลายผัดพริกเผา','หอยลาย, น้ำพริกเผา, กระเทียม, ใบโหระพา, น้ำมัน'),
 'Joke':('โจ๊ก','ข้าวต้มบด, หมูสับหรือเนื้อสัตว์, ขิง, ต้นหอม, ไข่ (บางสูตร)'),
 'KaiJeowMooSaap':('ไข่เจียวหมูสับ','ไข่, หมูสับ, น้ำปลา, น้ำมันทอด'),
 'KaiThoon':('ไข่ตุ๋น','ไข่, น้ำซุป, ซีอิ๊ว, หมูสับหรือผัก (บางสูตร)'),
 'KaoManGai':('ข้าวมันไก่','ข้าวหุงน้ำมันหรือมันไก่, เนื้อไก่, แตงกวา, น้ำซุป, ขิง, เต้าเจี้ยว, น้ำจิ้ม'),
 'KaoMooDang':('ข้าวหมูแดง','ข้าวสวย, หมูแดง, น้ำราด, แตงกวา, ไข่ต้ม (บางสูตร)'),
 'KhanomJeenNamYaKati':('ขนมจีนน้ำยากะทิ','เส้นขนมจีน, กะทิ, เนื้อปลา, เครื่องแกง, ผักเคียง'),
 'KhaoMokGai':('ข้าวหมกไก่','ข้าว, ไก่, เครื่องเทศ, หอมหัวใหญ่, โยเกิร์ตหรือกะทิ (ตามสูตร)'),
 'KhaoMooTodGratiem':('ข้าวหมูทอดกระเทียม','ข้าวสวย, เนื้อหมู, กระเทียม, พริกไทย, น้ำมัน'),
 'KhaoNiewMaMuang':('ข้าวเหนียวมะม่วง','ข้าวเหนียวมูน, กะทิ, น้ำตาล, เกลือ, มะม่วงสุก'),
 'KkaoKlukKaphi':('ข้าวคลุกกะปิ','ข้าว, กะปิ, หมูหวาน, ไข่ฝอย, มะม่วงเปรี้ยว, แตงกวา, ถั่วฝักยาว'),
 'KorMooYang':('คอหมูย่าง','คอหมู, น้ำปลา, กระเทียม, พริกไทย, น้ำจิ้มแจ่ว'),
 'KuaKling':('คั่วกลิ้ง','หมูสับหรือเนื้อสัตว์, พริกแกงคั่วกลิ้ง, ใบมะกรูด, ตะไคร้'),
 'KuayJab':('กวยจั๊บ','เส้นกวยจั๊บ, น้ำซุป, หมู, เครื่องใน (บางสูตร), ไข่'),
 'KuayTeowReua':('ก๋วยเตี๋ยวเรือ','เส้นก๋วยเตี๋ยว, หมูหรือเนื้อ, ลูกชิ้น, ผัก, น้ำซุปเครื่องเทศ'),
 'LarbMoo':('ลาบหมู','หมูสับ, ข้าวคั่ว, พริกป่น, น้ำปลา, มะนาว, หอมแดง, สะระแหน่'),
 'MassamanGai':('มัสมั่นไก่','ไก่, กะทิ, พริกแกงมัสมั่น, มันฝรั่ง, ถั่วลิสง, หอมหัวใหญ่'),
 'MooSatay':('หมูสะเต๊ะ','หมูหมักเครื่องเทศ, กะทิ, ซอสถั่วลิสง, อาจาด'),
 'NamTokMoo':('น้ำตกหมู','หมูย่าง, ข้าวคั่ว, พริกป่น, หอมแดง, มะนาว, น้ำปลา, สะระแหน่'),
 'PadPakBung':('ผัดผักบุ้ง','ผักบุ้ง, กระเทียม, พริก, ซอสปรุงรส, น้ำมัน'),
 'PadPakRuamMit':('ผัดผักรวมมิตร','ผักหลายชนิด, กระเทียม, น้ำมัน, ซอสปรุงรส, เนื้อสัตว์ (บางสูตร)'),
 'PadThai':('ผัดไทย','เส้นจันท์, เต้าหู้, ไข่, ถั่วงอก, กุยช่าย, ถั่วลิสง, น้ำมะขาม, กุ้ง (บางสูตร)'),
 'PadYordMala':('ผัดยอดมะระ','ยอดมะระ, กระเทียม, น้ำมัน, ซอสปรุงรส'),
 'PhatKaphrao':('ผัดกะเพรา','ใบกะเพรา, เนื้อสัตว์, พริก, กระเทียม, น้ำมัน, ซอสปรุงรส'),
 'PorkStickyNoodles':('ราดหน้าหมู','เส้นใหญ่หรือหมี่กรอบ, หมู, คะน้า, น้ำราดแป้ง, กระเทียม'),
 'Roast_duck':('เป็ดย่าง','เป็ด, เครื่องหมัก, น้ำราด, ผักเคียง'),
 'Roast_fish':('ปลาเผา','ปลา, เกลือ, ตะไคร้, ใบมะกรูด, น้ำจิ้ม'),
 'Somtam':('ส้มตำ','มะละกอดิบ, มะเขือเทศ, ถั่วฝักยาว, พริก, กระเทียม, มะนาว, น้ำปลา'),
 'SonInLawEggs':('ไข่ลูกเขย','ไข่ต้มทอด, น้ำมะขามเปียก, น้ำตาล, น้ำปลา, หอมเจียว'),
 'StewedPorkLeg':('ข้าวขาหมู','ข้าวสวย, ขาหมูพะโล้, ผักกาดดอง, คะน้า, น้ำจิ้ม'),
 'Suki':('สุกี้','ผักกาดขาว, วุ้นเส้น, ไข่, เนื้อสัตว์, น้ำจิ้มสุกี้'),
 'TomKhaGai':('ต้มข่าไก่','ไก่, กะทิ, ข่า, ตะไคร้, ใบมะกรูด, เห็ด, น้ำปลา, มะนาว'),
 'TomYumGoong':('ต้มยำกุ้ง','กุ้ง, ตะไคร้, ข่า, ใบมะกรูด, พริก, มะนาว, เห็ด'),
 'YamWoonSen':('ยำวุ้นเส้น','วุ้นเส้น, หมูสับหรือกุ้ง, หอมแดง, มะเขือเทศ, มะนาว, พริก'),
 'Yentafo':('เย็นตาโฟ','เส้นก๋วยเตี๋ยว, เต้าหู้ยี้หรือซอสเย็นตาโฟ, ลูกชิ้นปลา, ผักบุ้ง, หมึกกรอบ (บางสูตร)'),
}
assert len(MENU_METADATA)==50, f'เมนูไม่ครบ 50: {len(MENU_METADATA)}'
MENU_BY_TH = {v[0]: k for k,v in MENU_METADATA.items()}
print('รายชื่อเมนู:', len(MENU_METADATA),'ประเภท')
pd.DataFrame([{'ชื่ออังกฤษ':k,'ชื่อไทย':n,'ส่วนผสมทั่วไป':s} for k,(n,s) in MENU_METADATA.items()]).head(10)

รายชื่อเมนู: 50 ประเภท


,ชื่ออังกฤษ,ชื่อไทย,ส่วนผสมทั่วไป
0,BitterMelonSoup,ต้มมะระยัดไส้,"มะระ, หมูสับ, วุ้นเส้น (บางสูตร), กระเทียม, พร..."
1,BooPadPongali,ปูผัดผงกะหรี่,"เนื้อปู, ผงกะหรี่, ไข่, นม, หอมหัวใหญ่, ต้นหอม..."
2,CurriedFishCake,ทอดมันปลา,"เนื้อปลา, พริกแกงแดง, ถั่วฝักยาว, ใบมะกรูด, ไข..."
3,Dumpling,ขนมจีบ,"แผ่นเกี๊ยว, หมูหรือกุ้งสับ, กระเทียม, พริกไทย,..."
4,EggsStewed,ไข่พะโล้,"ไข่, หมู, เต้าหู้ (บางสูตร), โป๊ยกั๊ก, อบเชย, ..."
5,FriedChicken,ไก่ทอด,"ไก่, กระเทียม, พริกไทย, เกลือ, แป้ง (บางสูตร),..."
6,FriedKale,ผัดคะน้า,"คะน้า, กระเทียม, น้ำมัน, ซอสปรุงรส, เนื้อสัตว์..."
7,FriedMusselPancakes,หอยทอด,"หอยแมลงภู่, แป้ง, ไข่, ถั่วงอก, ต้นหอม, น้ำมัน"
8,GaengJued,แกงจืด,"น้ำซุป, ผักกาดขาวหรือสาหร่าย, เต้าหู้, หมูสับ,..."
9,GaengKeawWan,แกงเขียวหวาน,"กะทิ, พริกแกงเขียวหวาน, เนื้อสัตว์, มะเขือ, ใบ..."


## 2. ข้อมูลโภชนาการที่ตรวจสอบได้ + RAG จากแหล่งอ้างอิง

**แคลอรีไม่ได้อยู่ใน THFOOD-50** จึงไม่ใส่ตัวเลขให้ครบ 50 ด้วยการเดา: แสดงแคลอรีได้เมื่อพบหลักฐานที่ระบุ **ชื่อเมนู + หน่วยบริโภค** ตรงกัน หากยังไม่มีจะแจ้งอย่างสุภาพ และระบุว่าต้องใช้ปริมาณวัตถุดิบเพื่อประเมินเฉพาะจาน

แหล่งที่เตรียมไว้:
- [กรมอนามัย — ข้าวมันไก่ 558–619 กิโลแคลอรี/จาน](https://multimedia.anamai.moph.go.th/news/010265-2/)
- [กรมอนามัย — ข้าวเหนียวมะม่วง 200 กิโลแคลอรี สำหรับข้าวเหนียวมูล 50 กรัม + มะม่วงครึ่งผล](https://multimedia.anamai.moph.go.th/news/190465/)
- [กรมอนามัย — แนะนำลดหวาน มัน เค็ม กินอาหารหลากหลาย](https://multimedia.anamai.moph.go.th/news/120865-2/)

In [4]:
OFFICIAL_FACTS = {
 'ข้าวมันไก่': {
   'energy': 'ประมาณ 558–619 กิโลแคลอรี', 'portion': '1 จานตามตัวอย่างอ้างอิง',
   'source': 'กรมอนามัย', 'url':'https://multimedia.anamai.moph.go.th/news/010265-2/',
   'note':'พลังงานจานจริงเปลี่ยนตามขนาดข้าว เนื้อไก่ หนัง น้ำจิ้ม และน้ำมัน' },
 'ข้าวเหนียวมะม่วง': {
   'energy':'ประมาณ 200 กิโลแคลอรี', 'portion':'ข้าวเหนียวมูล 50 กรัม + มะม่วงสุกครึ่งผล (ไม่ใช่จานใหญ่)',
   'source':'กรมอนามัย','url':'https://multimedia.anamai.moph.go.th/news/190465/',
   'note':'ปริมาณกะทิ น้ำตาล และขนาดเสิร์ฟอาจทำให้พลังงานต่างไปมาก' }
}
OFFICIAL_DOCS = [
 {'id':'official:khaomangai','menu':'ข้าวมันไก่','title':'กรมอนามัย: ข้าวมันไก่และการปรับมื้ออาหาร',
  'text':'ข้าวมันไก่ 1 จาน พลังงาน 558-619 กิโลแคลอรี เปลี่ยนเป็นข้าวสวยลดเหลือ 505-539 กิโลแคลอรี ควรลดหนังไก่และน้ำจิ้มเพื่อหลีกเลี่ยงไขมันและโซเดียม',
  'url':'https://multimedia.anamai.moph.go.th/news/010265-2/','kind':'official'},
 {'id':'official:mango','menu':'ข้าวเหนียวมะม่วง','title':'กรมอนามัย: ข้าวเหนียวมะม่วง',
  'text':'ข้าวเหนียวมูล 50 กรัม พลังงาน 140 กิโลแคลอรี และมะม่วงสุกครึ่งผล 60 กิโลแคลอรี รวม 200 กิโลแคลอรีในหน่วยบริโภคตัวอย่าง ไม่ใช่ทั้งจาน',
  'url':'https://multimedia.anamai.moph.go.th/news/190465/','kind':'official'},
 {'id':'official:balance','menu':'','title':'กรมอนามัย: กินให้หลากหลายลดหวานมันเค็ม',
  'text':'แนะนำกินอาหารครบ 5 หมู่ เลือกอาหารต้ม นึ่ง อบ ย่าง เพิ่มผัก และลดหวาน มัน เค็ม โดยจัดสัดส่วนมื้ออาหารเหมาะสมกับกิจกรรม',
  'url':'https://multimedia.anamai.moph.go.th/news/120865-2/','kind':'official'},
]
# เพิ่มรายการพลังงานที่มีหลักฐานด้วยรูปแบบเดียวกับ OFFICIAL_FACTS เท่านั้น

In [5]:
# รองรับ PDF จริง (ไม่ต้องมี PDF ก็ทดลองตอบเรื่องเมนูได้)
import fitz
PDF_DIR = ROOT/'pdfs'; PDF_DIR.mkdir(exist_ok=True)

def import_pdf_files():
    try:
        from google.colab import files
    except ImportError:
        print('ใช้ไฟล์ใน',PDF_DIR,'ได้โดยตรงเมื่อไม่ได้รันใน Colab'); return
    print('เลือก PDF คู่มือโภชนาการ (กด Cancel ได้หากไม่มี)')
    uploaded = files.upload()
    for filename,data in uploaded.items():
        if filename.lower().endswith('.pdf') and data.startswith(b'%PDF'):
            (PDF_DIR/Path(filename).name).write_bytes(data)
    rebuild_rag()

def pdf_docs():
    result=[]
    for path in PDF_DIR.glob('*.pdf'):
        try:
            with fitz.open(path) as d:
                for page_id,page in enumerate(d):
                    text=page.get_text('text').strip()
                    if len(text)<40: continue
                    # เก็บข้อมูลหน้าจริง และแบ่งเป็นช่วงข้อความที่ไม่ซ้อนทับกัน
                    for j in range(0,len(text),1000):
                        seg=text[j:j+1000]
                        if len(seg)>40:
                            result.append({'id':f'pdf:{path.name}:{page_id+1}:{j}', 'menu':'',
                              'title':f'{path.name} หน้า {page_id+1}', 'text':seg,
                              'url':'', 'kind':'pdf'})
        except Exception as e: print('PDF อ่านไม่ได้:',path.name,str(e)[:100])
    return result

def rebuild_rag():
    global RAG_DOCS,RAG_VECTOR,RAG_MATRIX
    recipe_docs=[{'id':'recipe:'+class_en,'menu':name,'title':'ส่วนประกอบทั่วไป: '+name,
       'text':name+' มักมีส่วนประกอบ ได้แก่ '+ingredients+' (รายการตามสูตรทั่วไป ไม่ยืนยันว่ามีครบในภาพ)',
       'url':'','kind':'recipe'} for class_en,(name,ingredients) in MENU_METADATA.items()]
    RAG_DOCS=recipe_docs+OFFICIAL_DOCS+pdf_docs()
    RAG_VECTOR=TfidfVectorizer(analyzer='char',ngram_range=(2,4),min_df=1,sublinear_tf=True)
    RAG_MATRIX=RAG_VECTOR.fit_transform([r['title']+' '+r['text'] for r in RAG_DOCS])
    print('RAG พร้อม:',len(RAG_DOCS),'ข้อความ /',len(pdf_docs()),'ชังก์ PDF')

def retrieve(question, menu='', top_k=4):
    q=(menu+' ' if menu else '')+question
    scores=cosine_similarity(RAG_VECTOR.transform([q]),RAG_MATRIX)[0]
    if menu:
        for i,r in enumerate(RAG_DOCS):
            if r['menu']==menu: scores[i]+=0.5
    ix=np.argsort(scores)[::-1]; answer=[]; seen=set()
    for i in ix:
        d=RAG_DOCS[int(i)]
        ref=d['url'] or d['title']
        if ref in seen: continue # ไม่ใส่อ้างอิงลิงก์/หน้าเดิมซ้ำ
        seen.add(ref); answer.append({**d,'score':float(scores[i])})
        if len(answer)>=top_k: break
    return answer

rebuild_rag()

RAG พร้อม: 53 ข้อความ / 0 ชังก์ PDF


/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


## 3. แชทบอท: ใช้เมนูที่ AI จำแนกจากรูปโดยอัตโนมัติ

- คำถามที่ใช้คำว่า **“เมนูนี้”** จะอิงชื่อเมนูจากผลวิเคราะห์ภาพล่าสุดโดยตรง
- ตอบส่วนผสมทั่วไปได้ครบ 50 คลาส; ตัวเลขพลังงานมีเฉพาะเมนูที่มีแหล่งอ้างอิงและหน่วยบริโภค
- เปิด Gemini เพื่อช่วยสรุปจากบริบท RAG ได้ หากตั้งค่า `GEMINI_API_KEY` ใน Colab Secrets
- ไม่กล่าวอ้างว่าวิเคราะห์วัตถุดิบทั้งหมดหรือวัดแคลอรีจากภาพได้จริง และไม่ให้การวินิจฉัยทางการแพทย์


In [6]:
USE_GEMINI = False  # สลับเป็น True หลังตั้ง Secret ชื่อ GEMINI_API_KEY หากต้องการส่วนสรุปด้วย LLM

def get_gemini_key():
    if os.environ.get('GEMINI_API_KEY'): return os.environ['GEMINI_API_KEY']
    try:
        from google.colab import userdata
        return userdata.get('GEMINI_API_KEY') or ''
    except Exception: return ''

def ask_gemini(question, menu, retrieved):
    key=get_gemini_key()
    if not USE_GEMINI or not key: return ''
    try:
        from google import genai
        from google.genai import types
        context='\n\n'.join(f"[{d['id']}] {d['title']}: {d['text']}" for d in retrieved)
        prompt=("ตอบเป็นภาษาไทยอย่างตรงคำถามจากฐานความรู้ที่ให้เท่านั้น หากฐานความรู้ไม่ระบุแคลอรีห้ามแต่งตัวเลขพลังงาน ห้ามวินิจฉัยโรค "
                "ส่วนผสมแบบสูตรทั่วไปต้องบอกว่าอาจแตกต่างตามร้าน ตอบยาวไม่เกิน 180 คำ\n"
                f"ชื่อเมนู: {menu or 'ไม่ระบุ'}\nคำถาม: {question}\nข้อมูลค้นคืน:\n{context}")
        cli=genai.Client(api_key=key)
        resp=cli.models.generate_content(model='gemini-2.5-flash',contents=prompt,
          config=types.GenerateContentConfig(temperature=0.2))
        return (resp.text or '').strip()
    except Exception as e:
        print('Gemini ใช้งานไม่ได้; ใช้คำตอบในระบบแทน:',type(e).__name__)
        return ''

def ingredient_answer(menu):
    class_id=MENU_BY_TH.get(menu)
    if not class_id: return f'ยังไม่มีสูตรส่วนผสมที่ตรวจสอบได้สำหรับ **{menu}** ในฐาน 50 เมนู กรุณาเพิ่มข้อมูลสูตรที่มีแหล่งอ้างอิง'
    ingr=MENU_METADATA[class_id][1].split(', ')
    return '**ส่วนผสมทั่วไปของ '+menu+'**\n'+'\n'.join('- '+x for x in ingr)+ '\n\n*ส่วนผสมแตกต่างตามสูตร ร้าน และการดัดแปลงเมนู ไม่สามารถยืนยันจากภาพถ่ายเพียงอย่างเดียว*'

def energy_answer(menu):
    f=OFFICIAL_FACTS.get(menu)
    if f:
        return f"**พลังงานอ้างอิง:** {f['energy']} ต่อ {f['portion']}\n\n{f['note']}\n\nแหล่งอ้างอิง: [{f['source']}]({f['url']})"
    return ('**พลังงาน:** ยังไม่มีค่าที่ตรวจสอบแหล่งอ้างอิงและหน่วยบริโภคสำหรับเมนูนี้ในฐานข้อมูลปัจจุบัน '
            'จึงไม่เดาแคลอรีจากชื่อหรือภาพ หากระบุปริมาณและส่วนประกอบจริงพร้อมฐานข้อมูลพลังงานของวัตถุดิบ จึงจะคำนวณโดยประมาณได้')

def answer_question(question, menu='', use_llm=True):
    q=(question or '').strip()
    if not q: return 'พิมพ์คำถามที่ต้องการก่อนครับ'
    wants_ingredients=any(x in q for x in ['ส่วนผสม','วัตถุดิบ','ทำจากอะไร','ใส่อะไร'])
    wants_calories=any(x in q.lower() for x in ['แคล','kcal','พลังงาน','กิโลแคล'])
    wants_next=any(x in q for x in ['มื้อเย็น','มื้อถัดไป','มื้อต่อไป','กินอะไรดี','ควรเลือก','แนะนำอาหาร'])
    wants_med=any(x in q for x in ['เบาหวาน','ไต','ความดัน','โรคหัวใจ','ยา','รักษา','แพทย์'])
    found=retrieve(q,menu,top_k=5)
    if (wants_ingredients or wants_calories) and not menu:
        return 'กรุณาอัปโหลดภาพอาหารให้ AI วิเคราะห์ หรือกด **แก้ / เลือกเมนู** ก่อน เพื่อให้ตอบส่วนผสมและแคลอรีของเมนูที่ต้องการได้ตรงครับ'
    pieces=[]
    if menu: pieces.append(f'### 🍽️ เมนู: {menu}')
    if wants_ingredients: pieces.append(ingredient_answer(menu))
    if wants_calories: pieces.append(energy_answer(menu))
    if wants_next:
        pieces.append('**แนวทางมื้อถัดไป:** เพิ่มผัก เลือกโปรตีนไม่ติดมันและข้าวหรือแป้งในปริมาณพอดีกับกิจกรรม ลดเครื่องดื่มหวานและอาหารทอด ถ้ามื้อก่อนมีอาหารมันมาก มื้อต่อไปลองแกงจืด ผักลวกกับปลานึ่ง และข้าวในปริมาณเหมาะสม')
    if wants_med:
        pieces.append('**กรณีมีโรคประจำตัว:** ข้อมูลนี้เป็นคำแนะนำโภชนาการทั่วไป ไม่อาจใช้แทนนักกำหนดอาหารหรือแพทย์ที่ทราบผลตรวจ ยา และข้อจำกัดของผู้ใช้')
    if not (wants_ingredients or wants_calories or wants_next):
        summary=ask_gemini(q,menu,found) if use_llm else ''
        if summary: pieces.append(summary)
        else: pieces.append('อาหารที่เหมาะสมขึ้นอยู่กับเมนู ปริมาณที่รับประทาน และกิจกรรม ควรกินให้หลากหลายและลดหวาน มัน เค็ม หากต้องการทราบส่วนผสมหรือพลังงาน ให้ถามระบุเมนูหรือกดวิเคราะห์รูปอาหาร')
    elif wants_next and use_llm:
        summary=ask_gemini(q,menu,found)
        if summary: pieces.append('**คำอธิบายจากข้อมูลที่ค้นคืน:** '+summary)
    # เอกสารอ้างอิงใช้เอกสารจริงเท่านั้น; recipe card ไม่แสร้งเป็นแหล่งทางการ
    urls={}
    for d in found:
        if d['kind']=='official' and d['url']:
            urls[d['url']]=d['title']
    if urls:
        pieces.append('**เอกสารที่ค้นคืน:**\n'+'\n'.join('- ['+title+']('+url+')' for url,title in urls.items()))
    pieces.append('*ข้อจำกัด: ปริมาณแคลอรีไม่สามารถวัดได้แน่นอนจากรูปภาพ และข้อมูลนี้ไม่ใช่คำวินิจฉัยทางการแพทย์*')
    return '\n\n'.join(pieces)

# ทดสอบโดยไม่ต้องฝึกภาพก่อน
print(answer_question('เมนูนี้มีส่วนผสมอะไรบ้าง แคลเท่าไร','ข้าวมันไก่',use_llm=False)[:900])

### 🍽️ เมนู: ข้าวมันไก่

**ส่วนผสมทั่วไปของ ข้าวมันไก่**
- ข้าวหุงน้ำมันหรือมันไก่
- เนื้อไก่
- แตงกวา
- น้ำซุป
- ขิง
- เต้าเจี้ยว
- น้ำจิ้ม

*ส่วนผสมแตกต่างตามสูตร ร้าน และการดัดแปลงเมนู ไม่สามารถยืนยันจากภาพถ่ายเพียงอย่างเดียว*

**พลังงานอ้างอิง:** ประมาณ 558–619 กิโลแคลอรี ต่อ 1 จานตามตัวอย่างอ้างอิง

พลังงานจานจริงเปลี่ยนตามขนาดข้าว เนื้อไก่ หนัง น้ำจิ้ม และน้ำมัน

แหล่งอ้างอิง: [กรมอนามัย](https://multimedia.anamai.moph.go.th/news/010265-2/)

**เอกสารที่ค้นคืน:**
- [กรมอนามัย: ข้าวมันไก่และการปรับมื้ออาหาร](https://multimedia.anamai.moph.go.th/news/010265-2/)

*ข้อจำกัด: ปริมาณแคลอรีไม่สามารถวัดได้แน่นอนจากรูปภาพ และข้อมูลนี้ไม่ใช่คำวินิจฉัยทางการแพทย์*


## 4. โหลดรายชื่อ 50 คลาสจาก Hugging Face
รอบนี้โหลดเฉพาะ metadata ของ Dataset แบบ Streaming เพื่ออ่านชื่อคลาส ส่วนภาพจริงจะถูกดาวน์โหลด **ครั้งเดียวและแคชไว้บน Colab** ในขั้นเตรียมข้อมูล ไม่อ่านผ่านเน็ตซ้ำทุก Epoch

In [7]:
# แยก train, val, test ตามชุดที่ผู้เผยแพร่กำหนด ห้ามใช้ test ระหว่างเลือก hyperparameters
# รันเซลล์นี้ก่อนเทรน; จะต้องต่ออินเทอร์เน็ต
stream_train = load_dataset(HF_DATASET, split='train', streaming=True)
feature = stream_train.features['label']
CLASS_EN = list(feature.names)
assert len(CLASS_EN)==50 and set(CLASS_EN)==set(MENU_METADATA), (
    'รายชื่อคลาสจริงไม่ตรงกับแมปที่เตรียมไว้: '+str(set(CLASS_EN)^set(MENU_METADATA)))
IDX_TO_TH = [MENU_METADATA[n][0] for n in CLASS_EN]
print('Dataset:',HF_DATASET, 'จำนวนคลาส:',len(CLASS_EN))
print('ตัวอย่าง:',[(i,CLASS_EN[i],IDX_TO_TH[i]) for i in range(5)])

README.md:   0%|          | 0.00/4.28k [00:00<?, ?B/s]

Dataset: thean/THFOOD-50 จำนวนคลาส: 50
ตัวอย่าง: [(0, 'BitterMelonSoup', 'ต้มมะระยัดไส้'), (1, 'BooPadPongali', 'ปูผัดผงกะหรี่'), (2, 'CurriedFishCake', 'ทอดมันปลา'), (3, 'Dumpling', 'ขนมจีบ'), (4, 'EggsStewed', 'ไข่พะโล้')]


## 5. ตั้งค่าการฝึกแบบ Cached Features (รวดเร็วกว่าอ่านภาพทุกรอบ)
**ค่าแนะนำ:** ใช้ T4 GPU ก่อน, `FAST_DEV=True` เพื่อเช็กระบบอย่างรวดเร็ว แล้วเปลี่ยนเป็น `False` และรันใหม่เพื่อใช้ข้อมูลเต็มชุดในการรายงานผล

การแยก train/validation/test ยังคงตาม Dataset เดิม โดย Test Set จะถูกใช้เฉพาะตอนประเมินรอบสุดท้าย

In [8]:
# ===== ตั้งค่า =====
FAST_DEV = False  # True ใช้ตรวจ Pipeline เท่านั้น ห้ามนำคะแนนแบบนี้ไปอ้างเป็นผลเต็มชุด
HEAD_EPOCHS = 15  # นี่คือการฝึกเวกเตอร์ที่แคชไว้ ไม่ต้องวิ่ง EfficientNet ซ้ำ 15 รอบ
HEAD_BATCH_SIZE = 512
IMAGE_BATCH_SIZE = 64  # ขนาดแบตช์ตอนสกัดเวกเตอร์ด้วย CNN (หาก GPU ไม่พอ ลดเป็น 32)
NUM_WORKERS = 2       # ใน Colab CPU worker ช่วยอ่าน/แปลงภาพพร้อมกัน
IMG_SIZE = 224
HEAD_LR = 8e-4
EARLY_STOP_PATIENCE = 4
MAX_TRAIN = 2000 if FAST_DEV else None
MAX_VAL = 400 if FAST_DEV else None
MAX_TEST = 400 if FAST_DEV else None

from torch.utils.data import Dataset
from torch.utils.data.dataloader import default_collate
from time import perf_counter
from tqdm.auto import tqdm

valid_tf = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(), transforms.Normalize([.485,.456,.406],[.229,.224,.225])
])
# Feature cache ใช้ transformation แบบคงที่ทั้ง Train และ Validation เพื่อแคชหนึ่งครั้ง
train_tf = valid_tf

class FoodLocalDataset(Dataset):
    def __init__(self, split, transform, limit=None):
        # HF datasets แคชไฟล์ Arrow/ภาพที่ดาวน์โหลดไว้ใน runtime: ไม่สตรีมใหม่ทุกรอบ
        raw = load_dataset(HF_DATASET, split=split, streaming=False)
        self.raw = raw.select(range(min(limit,len(raw)))) if limit is not None else raw
        self.transform = transform
        self.split = split
    def __len__(self):
        return len(self.raw)
    def __getitem__(self, i):
        try:
            row = self.raw[int(i)]
            im = ImageOps.exif_transpose(row['image']).convert('RGB')
            return self.transform(im), int(row['label'])
        except (OSError, ValueError, TypeError) as ex:
            return None  # ไฟล์ภาพเสีย: ข้ามอย่างโปร่งใส แทนการใส่ label ปลอม

def _skip_bad(batch):
    valid=[x for x in batch if x is not None]
    return default_collate(valid) if valid else None

_DATASETS = {}
def loader(split, train=False, epoch=0):
    # Dataset instance reuse ไม่สร้าง object ใหม่ทุก epoch
    if split not in _DATASETS:
        cap = MAX_TRAIN if split=='train' else MAX_VAL if split=='val' else MAX_TEST
        print('เตรียมข้อมูล',split,'(ครั้งแรกอาจดาวน์โหลดข้อมูลจริง ใช้เวลาได้)')
        _DATASETS[split] = FoodLocalDataset(split, train_tf if split=='train' else valid_tf, cap)
    kwargs = {'batch_size': IMAGE_BATCH_SIZE, 'shuffle': bool(train),
              'num_workers': NUM_WORKERS, 'pin_memory': DEVICE=='cuda',
              'collate_fn': _skip_bad}
    if NUM_WORKERS > 0:
        kwargs.update({'persistent_workers': True, 'prefetch_factor': 2})
    return DataLoader(_DATASETS[split], **kwargs)

model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(CLASS_EN))
model.to(DEVICE)
for p in model.features.parameters():
    p.requires_grad_(False)
model.features.eval()  # คงสถิติ BatchNorm ของ pretrained model
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.classifier.parameters(), lr=HEAD_LR, weight_decay=1e-3)
print('โมเดล:',len(CLASS_EN),'คลาส | DEVICE:',DEVICE,'| FAST_DEV:',FAST_DEV)
if DEVICE!='cuda':
    print('⚠️ ไม่พบ GPU: Runtime → Change runtime type → T4 GPU ก่อนสกัดเวกเตอร์ จะเร็วกว่า CPU มาก')


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 129MB/s] 


โมเดล: 50 คลาส | DEVICE: cuda | FAST_DEV: False


## 6. สกัดคุณลักษณะภาพเพียงรอบเดียว แล้วฝึกตัวจำแนกจากแคช
- **ช่วงแรก:** เปิดภาพ Train/Validation และรัน EfficientNet ครั้งเดียวต่อภาพ (ส่วนนี้ยังใช้เวลาบ้าง และดาวน์โหลด Dataset ครั้งแรก)
- **ช่วงถัดไป:** ฝึก classifier จากเวกเตอร์ที่แคชไว้ ไม่อ่านภาพใหม่ทุก Epoch
- บันทึกเวลาจริงที่ใช้แยกแต่ละส่วน และบันทึกโมเดล Validation F1 ที่ดีที่สุด

หากรันใหม่โดยไม่ลบแคชเวกเตอร์ จะข้ามช่วงสกัดภาพได้เลย (แต่ Runtime ใหม่ของ Colab อาจไม่มีไฟล์แคชเดิม)

In [9]:
# ========= ประเมินภาพจริง (ใช้กับ Test Set หลังฝึก) =========
CHECKPOINT = ROOT / 'food_model_best.pth'

def evaluate_loader(model, dl):
    model.eval()
    truth, preds = [], []
    loss_sum, count = 0., 0
    with torch.inference_mode():
        for batch in dl:
            if batch is None:
                continue
            x,y=batch
            x=x.to(DEVICE, non_blocking=True)
            y=y.to(DEVICE, non_blocking=True)
            with torch.autocast('cuda',dtype=torch.float16,enabled=DEVICE=='cuda'):
                logits=model(x)
                loss=criterion(logits,y)
            loss_sum += float(loss.item())*len(y)
            count += len(y)
            truth.extend(y.cpu().tolist())
            preds.extend(logits.argmax(1).cpu().tolist())
    if not truth:
        raise RuntimeError('ไม่มีภาพที่อ่านได้สำหรับประเมิน')
    return {'loss':loss_sum/max(1,count), 'accuracy':accuracy_score(truth,preds),
            'macro_f1':f1_score(truth,preds,labels=list(range(len(CLASS_EN))),average='macro',zero_division=0),
            'true':truth,'pred':preds,'n':len(truth)}

# ========= 1) สกัดเวกเตอร์ CNN หนึ่งครั้งต่อภาพ =========
FEATURE_CACHE_VERSION = 1
cache_type='dev' if FAST_DEV else 'full'

def build_features(split):
    filename=ROOT / f'cached_features_{split}_{cache_type}_{IMG_SIZE}_v{FEATURE_CACHE_VERSION}.pt'
    if filename.exists():
        obj=torch.load(filename,map_location='cpu',weights_only=True)
        if obj.get('class_names')==CLASS_EN and obj.get('backbone')=='efficientnet_b0_imagenet1k':
            print('โหลดเวกเตอร์จากแคช:',filename.name,'จำนวน:',len(obj['y']))
            return obj['x'],obj['y']
    t0=perf_counter()
    model.features.eval()
    model.avgpool.eval()
    Z,Y=[],[]
    for batch in tqdm(loader(split,train=False),desc=f'Extract EfficientNet features [{split}]'):
        if batch is None:
            continue
        img,y=batch
        img=img.to(DEVICE,non_blocking=True)
        with torch.inference_mode():
            with torch.autocast('cuda',dtype=torch.float16,enabled=DEVICE=='cuda'):
                z=torch.flatten(model.avgpool(model.features(img)),1)
        Z.append(z.cpu().to(torch.float16 if DEVICE=='cuda' else torch.float32))
        Y.append(y.cpu())
    if not Z:
        raise RuntimeError('ไม่มีภาพให้สร้างเวกเตอร์')
    x,y=torch.cat(Z),torch.cat(Y)
    torch.save({'x':x,'y':y,'class_names':CLASS_EN,
        'backbone':'efficientnet_b0_imagenet1k'},filename)
    print(f'สร้างแคช {split}: {len(y)} ภาพ, {x.shape[1]} features, เวลา {perf_counter()-t0:.1f} วินาที')
    return x,y

extract_t0=perf_counter()
train_x, train_y = build_features('train')
val_x, val_y = build_features('val')
extract_seconds=perf_counter()-extract_t0

# ========= 2) Train เฉพาะ head ด้วยเวกเตอร์ที่แคช (ไม่รัน CNN ซ้ำ) =========
train_x,train_y=train_x.float().to(DEVICE),train_y.long().to(DEVICE)
val_x,val_y=val_x.float().to(DEVICE),val_y.long().to(DEVICE)
head=model.classifier
best=-1.
patience=0
training_log=[]
head_t0=perf_counter()
for epoch in range(HEAD_EPOCHS):
    head.train()
    perm=torch.randperm(len(train_y),device=DEVICE)
    running=0.
    for ids in perm.split(HEAD_BATCH_SIZE):
        optimizer.zero_grad(set_to_none=True)
        pred=head(train_x[ids])
        loss=criterion(pred,train_y[ids])
        loss.backward()
        optimizer.step()
        running+=float(loss.item())*len(ids)
    head.eval()
    with torch.inference_mode():
        vl=head(val_x)
        vloss=float(criterion(vl,val_y).item())
        yt=val_y.cpu().numpy()
        yp=vl.argmax(1).cpu().numpy()
    va=accuracy_score(yt,yp)
    vf=f1_score(yt,yp,labels=list(range(len(CLASS_EN))),average='macro',zero_division=0)
    row={'epoch':epoch+1,'train_loss':running/len(train_y),'val_loss':vloss,
         'val_accuracy':va,'val_macro_f1':vf,'train_n':len(train_y),'val_n':len(val_y)}
    training_log.append(row)
    print(f'Head epoch {epoch+1}/{HEAD_EPOCHS} | loss {row["train_loss"]:.4f} | val Accuracy {va:.3f} | Macro-F1 {vf:.3f}')
    if vf > best+1e-5:
        best=vf
        patience=0
        # ไฟล์ checkpoint ครบทั้ง EfficientNet และ classifier สำหรับใช้จริง
        torch.save({'model':model.state_dict(),'class_names':CLASS_EN,'class_th':IDX_TO_TH,
            'epoch':epoch+1,'best_val_macro_f1':best,'dataset':HF_DATASET,
            'fast_dev':FAST_DEV,'training_method':'cached_features_frozen_backbone'},CHECKPOINT)
    else:
        patience+=1
        if patience>=EARLY_STOP_PATIENCE:
            print('Early stopping: Validation Macro-F1 ไม่ดีขึ้น',EARLY_STOP_PATIENCE,'Epoch ติดต่อกัน')
            break
head_seconds=perf_counter()-head_t0
pd.DataFrame(training_log).to_csv(ROOT/'training_log.csv',index=False)
print(f'เวลาเตรียม/สกัด features (รวมโหลดแคช): {extract_seconds:.1f} วินาที')
print(f'เวลาฝึก classifier {len(training_log)} epochs: {head_seconds:.1f} วินาที')
print('Validation Macro-F1 ดีที่สุด:',best,'| บันทึกโมเดล:',CHECKPOINT)
print('หมายเหตุ: หาก Colab ต้องดาวน์โหลด Dataset รอบแรก จะใช้เวลามากกว่าเมื่อไฟล์อยู่ในแคช')


เตรียมข้อมูล train (ครั้งแรกอาจดาวน์โหลดข้อมูลจริง ใช้เวลาได้)


data/train-00000-of-00004-23c16aa1ab00b3(…): reconstructing file:   0%|          |  0.00B /  560MB            

data/train-00000-of-00004-23c16aa1ab00b3(…): downloading bytes:           |  0.00B            

data/train-00001-of-00004-d08e1d1df1c70f(…): reconstructing file:   0%|          |  0.00B /  563MB            

data/train-00001-of-00004-d08e1d1df1c70f(…): downloading bytes:           |  0.00B            

data/train-00002-of-00004-913bb8650d0188(…): reconstructing file:   0%|          |  0.00B /  560MB            

... (12 outputs left out)

สร้างแคช train: 12063 ภาพ, 1280 features, เวลา 224.1 วินาที
เตรียมข้อมูล val (ครั้งแรกอาจดาวน์โหลดข้อมูลจริง ใช้เวลาได้)


Extract EfficientNet features [val]:   0%|          | 0/25 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b900b87e020>
Traceback (most recent call last):
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b900b87e020>  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1706, in __del__

    Traceback (most recent call last):
self._shutdown_workers()  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1706, in __del__

  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1675, in _shutdown_workers
        if w.is_alive():self._shutdown_workers()

  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1675, in _shutdown_workers
        if w.is_alive():assert self._parent_pid == os.getpid(), 'can only test a child process'

AssertionError  File "/usr/lib/python3.13/multiprocessing/proce

สร้างแคช val: 1600 ภาพ, 1280 features, เวลา 24.5 วินาที
Head epoch 1/15 | loss 3.1732 | val Accuracy 0.537 | Macro-F1 0.432
Head epoch 2/15 | loss 2.0175 | val Accuracy 0.689 | Macro-F1 0.649
Head epoch 3/15 | loss 1.4592 | val Accuracy 0.744 | Macro-F1 0.712
Head epoch 4/15 | loss 1.1752 | val Accuracy 0.773 | Macro-F1 0.753
Head epoch 5/15 | loss 1.0082 | val Accuracy 0.787 | Macro-F1 0.772
Head epoch 6/15 | loss 0.8969 | val Accuracy 0.803 | Macro-F1 0.792
Head epoch 7/15 | loss 0.8124 | val Accuracy 0.809 | Macro-F1 0.798
Head epoch 8/15 | loss 0.7515 | val Accuracy 0.815 | Macro-F1 0.803
Head epoch 9/15 | loss 0.6988 | val Accuracy 0.821 | Macro-F1 0.809
Head epoch 10/15 | loss 0.6547 | val Accuracy 0.823 | Macro-F1 0.810
Head epoch 11/15 | loss 0.6212 | val Accuracy 0.830 | Macro-F1 0.820
Head epoch 12/15 | loss 0.5883 | val Accuracy 0.831 | Macro-F1 0.819
Head epoch 13/15 | loss 0.5627 | val Accuracy 0.834 | Macro-F1 0.823
Head epoch 14/15 | loss 0.5346 | val Accuracy 0.833 | Ma

## 7. ประเมินภาพจริงจาก Test Set (รันครั้งเดียวหลังฝึก)
นำโมเดล EfficientNet + classifier ที่บันทึกไว้ไปวิเคราะห์ภาพ Test **จริง** ทุกภาพ แล้วคำนวณ Accuracy, Macro F1, Confusion Matrix (ไม่มีคะแนนสมมติ)

**สำคัญ:** เวอร์ชันนี้ฝึกเฉพาะหัวจำแนกจาก features ที่แคชไว้ ความแม่นยำอาจยังไม่ดีพอในการใช้งานจริง ต้องตรวจผลรายคลาสและตัวอย่างภาพผิดก่อนเผยแพร่

In [10]:
assert CHECKPOINT.exists(), 'ต้องฝึกโมเดลและเซฟ Checkpoint ให้สำเร็จก่อน'
checkpoint=torch.load(CHECKPOINT,map_location=DEVICE,weights_only=False)
assert checkpoint['class_names']==CLASS_EN
model.load_state_dict(checkpoint['model']);model.eval()
metrics=evaluate_loader(model,loader('test'))
print(f"Test n={metrics['n']}, Accuracy={metrics['accuracy']:.4f}, Macro-F1={metrics['macro_f1']:.4f}")
rep=classification_report(metrics['true'],metrics['pred'],labels=list(range(50)),target_names=IDX_TO_TH,
 output_dict=True,zero_division=0)
pd.DataFrame(rep).transpose().to_csv(ROOT/'test_classification_report.csv',encoding='utf-8-sig')
pd.DataFrame(confusion_matrix(metrics['true'],metrics['pred'],labels=list(range(50))),
 index=IDX_TO_TH,columns=IDX_TO_TH).to_csv(ROOT/'confusion_matrix_50.csv',encoding='utf-8-sig')
result={'dataset':HF_DATASET,'split':'test','n':metrics['n'],'fast_dev':FAST_DEV,
 'accuracy':float(metrics['accuracy']),'macro_f1':float(metrics['macro_f1']),
 'best_val_macro_f1':float(checkpoint['best_val_macro_f1']),
 'classes':50,'checkpoint':str(CHECKPOINT),
 'training_method':checkpoint.get('training_method','unknown')}
(ROOT/'evaluation.json').write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding='utf-8')
print('สร้างไฟล์ผลประเมิน:',list(ROOT.glob('*report*')))

เตรียมข้อมูล test (ครั้งแรกอาจดาวน์โหลดข้อมูลจริง ใช้เวลาได้)
Test n=2105, Accuracy=0.8375, Macro-F1=0.8231
สร้างไฟล์ผลประเมิน: [PosixPath('/content/ThaiFoodAI_THFOOD50/test_classification_report.csv')]


## 8. วิเคราะห์ภาพอาหารอัตโนมัติ

อัปโหลดภาพ → โมเดลเลือกผลทำนายอันดับแรกจาก 50 เมนูโดยอัตโนมัติ → แสดงชื่อและคะแนนความมั่นใจ → ส่งชื่อเมนูนี้ให้แชทใช้ต่อทันที

**ไม่แสดงปุ่มแก้ชื่อเมนู กล่องเลือกเมนู หรือช่องพิมพ์ชื่อเมนูเอง** หากไม่มั่นใจให้แสดงคำเตือนพร้อม 5 อันดับแรกเท่านั้น โดยไม่เปลี่ยนผลทำนายเอง หากยังไม่มี checkpoint ให้แจ้งว่าต้องฝึกโมเดลก่อน


In [11]:
# โหลด checkpoint ที่ฝึกจริงเท่านั้น; ห้ามเปิดโหมดเลือกเมนูทดแทน AI
MODEL_READY = (ROOT/'food_model_best.pth').exists()
if MODEL_READY:
    ck=torch.load(ROOT/'food_model_best.pth',map_location=DEVICE,weights_only=False)
    class_for_model=ck['class_names']
    # โมเดลจากเซลล์ 5 อยู่แล้ว; ถ้ารันเซลล์นี้ใหม่ให้สร้าง classifier จาก metadata
    if 'model' not in globals():
        model=models.efficientnet_b0(weights=None)
        model.classifier[1]=nn.Linear(model.classifier[1].in_features,len(class_for_model))
        model=model.to(DEVICE)
    model.load_state_dict(ck['model']);model.eval()
else:
    class_for_model=list(MENU_METADATA)
    print('ยังไม่มีโมเดลที่ฝึก: กรุณาฝึก EfficientNet และบันทึก checkpoint ก่อนวิเคราะห์ภาพ')

@torch.inference_mode()
def predict_food(image):
    if image is None: return []
    if not MODEL_READY: return []
    x=valid_tf(ImageOps.exif_transpose(image).convert('RGB')).unsqueeze(0).to(DEVICE)
    prob=torch.softmax(model(x),dim=1).cpu().flatten()
    values,idx=prob.topk(min(5,len(class_for_model)))
    return [(MENU_METADATA[class_for_model[int(i)]][0],float(v)) for v,i in zip(values,idx)]

## 9. ประเมิน RAG (Retrieval Recall@3 / MRR@3) เบื้องต้น
ตัวอย่างคำถามนี้เป็น **dev smoke test ที่ผู้พัฒนาสร้างเอง** สำหรับตรวจการค้นคืน ไม่ใช่ผลที่อ้างความแม่นยำกับผู้ใช้จริง ต้องเก็บคำถามใหม่และทำ gold relevance จากผู้ประเมินภายหลัง

In [12]:
rag_test=[
 ('ข้าวมันไก่ มีแคลอรีเท่าไร','ข้าวมันไก่','official:khaomangai'),
 ('ข้าวเหนียวมะม่วง 50 กรัม และมะม่วงครึ่งลูก','ข้าวเหนียวมะม่วง','official:mango'),
 ('ข้าวมันไก่ใส่ส่วนผสมอะไรบ้าง','ข้าวมันไก่','recipe:KaoManGai'),
 ('ผัดกะเพราใช้วัตถุดิบอะไร','ผัดกะเพรา','recipe:PhatKaphrao'),
 ('ส้มตำมีส่วนผสมอะไร','ส้มตำ','recipe:Somtam'),
 ('ลดหวานมันเค็มอย่างไร','','official:balance'),
]
rows=[]
for q,menu,gold in rag_test:
    ids=[d['id'] for d in retrieve(q,menu,top_k=3)]
    pos=(ids.index(gold)+1) if gold in ids else None
    rows.append({'question':q,'menu':menu,'gold':gold,'retrieved_ids':str(ids),
                 'recall_at_3':int(pos is not None),'rr_at_3':(1/pos if pos else 0)})
rag_report=pd.DataFrame(rows)
rag_report.to_csv(ROOT/'rag_retrieval_dev.csv',index=False,encoding='utf-8-sig')
print('Dev Recall@3:',rag_report.recall_at_3.mean(),'MRR@3:',rag_report.rr_at_3.mean())
display(rag_report)

Dev Recall@3: 1.0 MRR@3: 0.8333333333333334


,question,menu,gold,retrieved_ids,recall_at_3,rr_at_3
0,ข้าวมันไก่ มีแคลอรีเท่าไร,ข้าวมันไก่,official:khaomangai,"['recipe:KaoManGai', 'official:khaomangai', 'o...",1,0.5
1,ข้าวเหนียวมะม่วง 50 กรัม และมะม่วงครึ่งลูก,ข้าวเหนียวมะม่วง,official:mango,"['recipe:KhaoNiewMaMuang', 'official:mango', '...",1,0.5
2,ข้าวมันไก่ใส่ส่วนผสมอะไรบ้าง,ข้าวมันไก่,recipe:KaoManGai,"['recipe:KaoManGai', 'official:khaomangai', 'r...",1,1.0
3,ผัดกะเพราใช้วัตถุดิบอะไร,ผัดกะเพรา,recipe:PhatKaphrao,"['recipe:PhatKaphrao', 'recipe:FriedKale', 're...",1,1.0
4,ส้มตำมีส่วนผสมอะไร,ส้มตำ,recipe:Somtam,"['recipe:Somtam', 'recipe:Suki', 'recipe:Bitte...",1,1.0
5,ลดหวานมันเค็มอย่างไร,,official:balance,"['official:balance', 'recipe:GaengKeawWan', 'r...",1,1.0


## 10. หน้าเว็บ Gradio — วิเคราะห์ภาพอัตโนมัติ / แชท RAG / Responsible AI

- อัปโหลดรูปอาหารแล้วกด **วิเคราะห์รูปอาหาร** → ระบบทายชื่ออัตโนมัติจากโมเดลที่ฝึกจริง
- แสดงชื่อเมนู คะแนนความมั่นใจ และ 5 อันดับแรกเพื่อความโปร่งใส **โดยไม่มีปุ่มหรือช่องแก้/เลือกเมนู**
- แชทใช้ผลจำแนกภาพล่าสุดโดยอัตโนมัติ เช่น ถาม “เมนูนี้มีส่วนผสมอะไรบ้าง กี่แคล?”
- ถ้ายังไม่มีโมเดล ให้แจ้งว่าต้องฝึกก่อน ไม่แสดงคำทำนายปลอม


In [13]:
import gradio as gr

def menu_details(menu):
    if not menu:
        return 'ยังไม่มีผลการวิเคราะห์อาหาร กรุณาอัปโหลดภาพแล้วกดวิเคราะห์รูปอาหาร'
    return answer_question('เมนูนี้มีส่วนผสมอะไรบ้าง แคลอรีเท่าไร', menu, use_llm=False)

def do_predict(image):
    """ทายจากโมเดลที่ฝึกจริงเท่านั้น ไม่รับและไม่เสนอการแก้ชื่อเมนูด้วยตนเอง"""
    empty_state = {'menu': None, 'confidence': None, 'mode': 'none'}
    if image is None:
        return 'กรุณาอัปโหลดภาพอาหารก่อน', 'ยังไม่มีข้อมูล', empty_state
    if not MODEL_READY:
        return ('**ยังไม่มีโมเดลที่ฝึกเสร็จ** กรุณารันเซลล์ฝึกและประเมินโมเดลก่อนใช้งาน',
                'ยังไม่สามารถแสดงส่วนผสมหรือแคลอรีจากภาพได้', empty_state)
    try:
        preds = predict_food(image)
    except Exception as exc:
        return (f'วิเคราะห์ภาพไม่สำเร็จ: {type(exc).__name__}',
                'กรุณาตรวจสอบโมเดลและลองใหม่', empty_state)
    if not preds:
        return 'โมเดลไม่สามารถวิเคราะห์ภาพนี้ได้', 'ยังไม่มีข้อมูล', empty_state
    menu, confidence = preds[0]
    gap = confidence - preds[1][1] if len(preds) > 1 else confidence
    uncertain = confidence < 0.60 or gap < 0.10
    ranking = '\n'.join(f'- {n}: {p:.1%}' for n, p in preds)
    result = (f'### ผลจาก AI: **{menu}**\n'
              f'คะแนน Softmax: {confidence:.1%} (ไม่ใช่การรับประกันความถูกต้อง)\n\n'
              f'**5 อันดับแรก:**\n{ranking}')
    if uncertain:
        result += ('\n\n⚠️ **ผลทำนายมีความไม่แน่นอน** โมเดลอาจจำแนกผิด '
                   'และไม่สามารถยืนยันตัวตนของอาหารจากภาพเพียงอย่างเดียว')
    return result, menu_details(menu), {'menu': menu, 'confidence': confidence, 'mode': 'predicted'}

def chat_send(message, history, state):
    history = list(history or [])
    if not (message or '').strip():
        return history, ''
    menu = (state or {}).get('menu') or ''
    history.append({'role': 'user', 'content': message})
    history.append({'role': 'assistant', 'content': answer_question(message, menu)})
    return history, ''

def reset_after_image_change():
    return ('ภาพใหม่พร้อมวิเคราะห์ กรุณากดวิเคราะห์รูปอาหาร',
            'ยังไม่ได้วิเคราะห์ภาพใหม่นี้',
            {'menu': None, 'confidence': None, 'mode': 'none'})

with gr.Blocks(title='ThaiFood AI | THFOOD-50', theme=gr.themes.Soft()) as demo:
    gr.Markdown('# 🍛 ThaiFood AI — วิเคราะห์อาหารไทยจากภาพ พร้อมแชทโภชนาการ')
    gr.Markdown('**จำแนกอัตโนมัติจากภาพด้วยโมเดล 50 คลาส** • แสดงส่วนผสมตามสูตรทั่วไป • แคลอรีเฉพาะที่มีแหล่งข้อมูลอ้างอิง')
    state = gr.State({'menu': None, 'confidence': None, 'mode': 'none'})
    with gr.Tab('📷 วิเคราะห์ภาพอาหาร'):
        with gr.Row():
            img = gr.Image(label='ถ่ายภาพ / อัปโหลดภาพอาหาร', type='pil', sources=['upload', 'webcam'])
            with gr.Column():
                btn = gr.Button('วิเคราะห์รูปอาหาร', variant='primary')
                output = gr.Markdown('อัปโหลดภาพแล้วกดวิเคราะห์')
        detail = gr.Markdown('เมื่อวิเคราะห์รูปแล้ว จะแสดงข้อมูลส่วนผสมและพลังงานอ้างอิงที่มี')
    with gr.Tab('💬 ถามเรื่องอาหารและโภชนาการ'):
        gr.Markdown('**ตัวอย่าง:** “เมนูนี้มีส่วนผสมอะไรบ้าง แคลเท่าไร”, “กินเมนูนี้แล้วมื้อเย็นควรกินอะไร?” บอทจะอิงเมนูที่ AI ทายจากภาพล่าสุด')
        import inspect
        _chat_args = {'height': 470, 'label': 'ThaiFood AI Chat'}
        if 'type' in inspect.signature(gr.Chatbot.__init__).parameters:
            _chat_args['type'] = 'messages'
        chatbot = gr.Chatbot(**_chat_args)
        with gr.Row():
            message = gr.Textbox(placeholder='ถามเรื่องส่วนผสม แคลอรี หรือการเลือกอาหาร...', label='คำถาม', scale=5)
            send = gr.Button('ส่ง', variant='primary', scale=1)
    with gr.Tab('📊 การวัดผล AI'):
        if (ROOT / 'evaluation.json').exists():
            gr.JSON(value=json.loads((ROOT / 'evaluation.json').read_text()), label='ผลทดสอบจาก Test Set')
        else:
            gr.Markdown('ยังไม่ได้ประเมิน Test Set จริง: ต้องรันเซลล์ฝึกและทดสอบก่อน **ไม่มีการสร้างคะแนนสมมติ**')
        gr.Markdown(f"**RAG development smoke test:** Recall@3={rag_report['recall_at_3'].mean():.3f}, MRR@3={rag_report['rr_at_3'].mean():.3f} จาก {len(rag_report)} คำถามที่ผู้พัฒนาสร้างเอง; ไม่ใช่การประเมินอิสระ")
    with gr.Tab('🛡️ Responsible AI'):
        gr.Markdown("""### ข้อจำกัด
1. โมเดลรู้จักเฉพาะ 50 ประเภทใน Dataset และอาจจำแนกผิด โดยเฉพาะอาหารนอกชุดข้อมูล
2. รูปอาหารไม่สามารถบอกน้ำหนักจริง ปริมาณน้ำมัน เครื่องปรุง และส่วนผสมทั้งหมดได้
3. ส่วนผสมเป็นสูตรทั่วไป ไม่ใช่หลักฐานว่าจานในภาพมีครบทุกชนิด
4. ตัวเลขแคลอรีเป็นข้อมูลต่อหน่วยบริโภคจากแหล่งอ้างอิง **ไม่ใช่การวัดจากภาพ** เมนูที่ไม่มีแหล่งข้อมูลจะไม่สร้างตัวเลขขึ้นเอง
5. คำแนะนำไม่แทนแพทย์หรือนักกำหนดอาหาร โดยเฉพาะผู้ที่มีโรคประจำตัว
6. ควรขอความยินยอมในการเก็บภาพ และไม่เปิดเผยข้อมูลส่วนบุคคล""")
    img.change(reset_after_image_change, inputs=[], outputs=[output, detail, state])
    btn.click(do_predict, inputs=[img], outputs=[output, detail, state])
    send.click(chat_send, inputs=[message, chatbot, state], outputs=[chatbot, message])
    message.submit(chat_send, inputs=[message, chatbot, state], outputs=[chatbot, message])

demo.launch(share=True, debug=False)


/tmp/ipykernel_707/2090616060.py:49: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title='ThaiFood AI | THFOOD-50', theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a363dd3e21442734a0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 11. ส่งไฟล์โครงงานและข้อมูลจริงให้ผู้สอน

- **Source Code:** Notebook `.ipynb` นี้
- **Image Dataset:** https://huggingface.co/datasets/thean/THFOOD-50 (50 ประเภท; ตรวจสิทธิใช้งานจากหน้า Dataset ก่อนเผยแพร่ซ้ำ)
- **โภชนาการ:** หลักฐานใน `OFFICIAL_FACTS` และลิงก์แหล่งอ้างอิง (แยกจาก Dataset ภาพ)
- **ตัวเลขวัดผล:** `evaluation.json`, `training_log.csv`, `test_classification_report.csv`, `confusion_matrix_50.csv` หลังรันจริง
- **Demo:** `share=True` ให้ลิงก์ชั่วคราว; ลิงก์ถาวรให้ Deploy บน Hugging Face Spaces
- **ข้อจำกัด:** ระบบไม่เปิดให้เลือกชื่อเมนูเอง และอาจทายผิดเมื่อภาพนอกคลาส; บอทจะใช้ผลทำนายจาก AI เท่านั้นในคำถามที่อ้างถึง “เมนูนี้”


In [14]:
# ดาวน์โหลดผลประเมินที่สร้างจาก Colab เป็น ZIP (ไม่รวม Dataset ขนาดหลาย GB)
import zipfile
report_zip=ROOT/'submission_reports.zip'
with zipfile.ZipFile(report_zip,'w',zipfile.ZIP_DEFLATED) as z:
    for name in ['evaluation.json','training_log.csv','test_classification_report.csv',
                 'confusion_matrix_50.csv','rag_retrieval_dev.csv']:
        f=ROOT/name
        if f.exists():z.write(f,arcname=name)
print('ไฟล์รายงาน:',report_zip)
try:
    from google.colab import files
    files.download(str(report_zip))
except ImportError: pass

ไฟล์รายงาน: /content/ThaiFoodAI_THFOOD50/submission_reports.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>